In [1]:
# Cell 1: imports & global config
import random
import json
from typing import List, Dict, Any

import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

# === Configure global parameters ===
K_SHOTS = [0, 3, 5, 10]   # Selected K value
MAX_INPUT_LENGTH = 512
MAX_OUTPUT_LENGTH = 256
NUM_BEAMS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GLOBAL_RANDOM_SEED = 42   # Set random seed for reproducibility
random.seed(GLOBAL_RANDOM_SEED)

In [ ]:
# Cell 2: load test set

test_csv_path = "Dataset/MolGen_Test.csv"
df = pd.read_csv(test_csv_path)
captions = df["Caption"].tolist()
true_smiles = df["SMILES"].tolist()

len(captions), df.head()

(60,
                                       SMILES  \
 0             C(C#CC1=CC=NC=C1)#CC1=CC=NC=C1   
 1             C(C#CC1=CC=NC=C1)#CC1=CC=NC=C1   
 2             C(C#CC1=CC=NC=C1)#CC1=CC=NC=C1   
 3  C1=CC(=CC=C1C#CC2=CC=C(C=C2)C(=O)O)C(=O)O   
 4  C1=CC(=CC=C1C#CC2=CC=C(C=C2)C(=O)O)C(=O)O   
 
                                              Caption  
 0  This molecule has two pyridine cores linked by...  
 1  This molecule uses a diacetylene chain to link...  
 2  This molecule connects two pyridine rings via ...  
 3  This molecule consists of a diphenylacetylene ...  
 4  This molecule features a diphenylacetylene bac...  )

In [ ]:
# Cell 3: load model

model_path = "YOUR_MODEL_PATH/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_path, model_max_length=MAX_INPUT_LENGTH, legacy=False)
model = AutoModelForCausalLM.from_pretrained(model_path)
model.to(DEVICE)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-05)
    (

In [ ]:
# Cell 4: Load prompt examples from JSON file

def load_prompt_examples_from_json(json_file_path: str) -> List[Dict[str, Any]]:
    with open(json_file_path, "r") as f:
        prompt_examples = json.load(f)
    return prompt_examples

prompt_json_path = "Dataset/MolGen_prompts.json"
prompt_examples = load_prompt_examples_from_json(prompt_json_path)

len(prompt_examples), prompt_examples[0]

(266, {'descriptions': ['This molecule is benzene.'], 'smiles': 'C1=CC=CC=C1'})

In [5]:
# Cell 5: System Prompt（Cap2Mol）

SYSTEM_PROMPT = """You are now working as an excellent expert in single-molecule discovery.

Given the caption of a molecule, your job is to predict the SMILES representation of the molecule.
The caption describes the molecule structures using skeleton and anchor groups.
You must infer the correct SMILES representation from the caption.

Output rule:
Output only one SMILES string.
"""

print(SYSTEM_PROMPT)

You are now working as an excellent expert in single-molecule discovery.

Given the caption of a molecule, your job is to predict the SMILES representation of the molecule.
The caption describes the molecule structures using skeleton and anchor groups.
You must infer the correct SMILES representation from the caption.

Output rule:
Output only one SMILES string.



In [6]:
# Cell 6: Construct few-shot example

def build_fewshot_examples(prompt_examples: List[Dict[str, Any]], k_shot: int) -> str:
    """
    Randomly select `k_shot` examples from `prompt_examples`
    and concatenate them into a Cap2Mol few-shot prompt.
    - Returns an empty string if `k_shot == 0`(zero-shot).
    - Uses all available examples if `k_shot >= len(prompt_examples)`.
    """
    if k_shot <= 0:
        return ""  # zero-shot

    if k_shot >= len(prompt_examples):
        selected = prompt_examples
    else:
        selected = random.sample(prompt_examples, k_shot)

    blocks = []
    for idx, ex in enumerate(selected, start=1):
        caption_text = ex["descriptions"][0]
        smiles = ex["smiles"]

        block = f"""Example {idx}:
Instruction: Given the caption of a molecule, predict the SMILES representation.
Input:
{caption_text}
Output:
{smiles}
"""
        blocks.append(block)

    return "\n".join(blocks)

# Verify few-shot example output style
print(build_fewshot_examples(prompt_examples, k_shot=2))

Example 1:
Instruction: Given the caption of a molecule, predict the SMILES representation.
Input:
This molecule is composed of a straight carbon chain containing eight carbon atoms and is terminated by sulfhydryl groups on both sides serving as anchoring groups.
Output:
C(CCCCS)CCCS

Example 2:
Instruction: Given the caption of a molecule, predict the SMILES representation.
Input:
This molecule is Diethynylbenzene.
Output:
C#CC1=CC=C(C=C1)C#C



In [7]:
# Cell 7: Assemble final prompt（System + few-shot + Query）

def build_prompt_for_caption(caption: str,
                             prompt_examples: List[Dict[str, Any]],
                             k_shot: int) -> str:
    """
    Construct the final prompt for model input, consisting of:
    - System prompt (task definition + output JSON format rules)
    - A set of few-shot examples (Example i)
    - Current query (caption for which SMILES is to be generated)
    """
    system_part = SYSTEM_PROMPT
    fewshot_part = build_fewshot_examples(prompt_examples, k_shot=k_shot)

    if fewshot_part:
        fewshot_part = fewshot_part + "\n"

    query_part = f"""
Now, solve the following query:

Instruction: Given the caption of a molecule, predict the SMILES representation.
Input:
{caption}
Output:
"""

    full_prompt = system_part + fewshot_part + query_part
    return full_prompt

# Verify final prompt
test_prompt = build_prompt_for_caption(captions[0], prompt_examples, k_shot=3)
print(test_prompt)

You are now working as an excellent expert in single-molecule discovery.

Given the caption of a molecule, your job is to predict the SMILES representation of the molecule.
The caption describes the molecule structures using skeleton and anchor groups.
You must infer the correct SMILES representation from the caption.

Output rule:
Output only one SMILES string.
Example 1:
Instruction: Given the caption of a molecule, predict the SMILES representation.
Input:
This molecule has a structure of three linked benzene rings between two pyridine rings, with nitrogen atoms in the pyridine rings as potential anchoring groups.
Output:
C1=CC(=CC=C1C2=CC=C(C=C2)C3=CC=NC=C3)C4=CC=C(C=C4)C5=CC=NC=C5

Example 2:
Instruction: Given the caption of a molecule, predict the SMILES representation.
Input:
This molecule is pyrene.
Output:
C1=CC2=C3C(=C1)C=CC4=CC=CC(=C43)C=C2

Example 3:
Instruction: Given the caption of a molecule, predict the SMILES representation.
Input:
This molecule contains a chain of t

In [ ]:
# Cell 8: Generate a single SMILES string and extract the "molecule" field

import re

# Character set commonly found in SMILES strings
_SMILES_RE = re.compile(r"[A-Za-z0-9@+\-\[\]\(\)=#$\\/%.]+")  # Excludes whitespace

def _strip_wrappers(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\\boxed\{([^}]*)\}", r"\1", s).strip()
    # Remove surrounding $ (LaTeX math mode)
    s = s.strip("$").strip()
    # Remove quotes and backticks
    s = s.strip('`"\'')
    # Remove trailing punctuation
    s = re.sub(r"[.,;:]+$", "", s)
    return s

def _looks_like_smiles_minimal(s: str) -> bool:
    s = s.strip()
    if len(s) < 3:
        return False
    return bool(re.search(r"[=#\(\)\[\]0-9]", s))

def extract_smiles(raw_text: str) -> str:
    t = raw_text.strip()
    # 0) Search for a SMILES-like candidate in the first few lines after "Output:".
    m = re.search(r"Output:\s*(.*)", t, flags=re.IGNORECASE | re.DOTALL)
    if m:
        tail = m.group(1)
        lines = [ln.strip() for ln in tail.splitlines() if ln.strip()][:5]
        for ln in lines:
            ln = _strip_wrappers(ln).replace(" ", "")
            m2 = _SMILES_RE.search(ln)
            if m2:
                cand = _strip_wrappers(m2.group(0))
                if _looks_like_smiles_minimal(cand):
                    return cand
        # If no valid candidate is found in the "Output:" section, proceed to other strategies.

    # 1) Handle patterns like "final answer is: ..."
    m = re.search(r"final answer\s*(is|:)\s*([^\n\r]+)", t, flags=re.IGNORECASE)
    if m:
        cand = _strip_wrappers(m.group(2)).replace(" ", "")
        m2 = _SMILES_RE.search(cand)
        if m2:
            cand2 = _strip_wrappers(m2.group(0))
            if _looks_like_smiles_minimal(cand2):
                return cand2

    # 2) Extract content from \boxed{...}
    m = re.search(r"\\boxed\{([^}]*)\}", t)
    if m:
        cand = _strip_wrappers(m.group(1)).replace(" ", "")
        m2 = _SMILES_RE.search(cand)
        if m2:
            cand2 = _strip_wrappers(m2.group(0))
            if _looks_like_smiles_minimal(cand2):
                return cand2

    # 3) Check the first line of output as a potential SMILES candidate
    first_line = t.splitlines()[0].strip() if t else ""
    m = _SMILES_RE.search(first_line)
    if m:
        cand = _strip_wrappers(m.group(0))
        if _looks_like_smiles_minimal(cand):
            return cand

    # 4) Scan all tokens and select the most plausible SMILES candidate
    cands = _SMILES_RE.findall(t)
    if not cands:
        return ""

    # Filter out known non-SMILES tokens
    bad = {"You", "Example", "Instruction", "Input", "Output", "Step", "Explanation", "final", "answer"}
    cleaned = []
    for c in cands:
        c2 = _strip_wrappers(c)
        if not c2 or c2 in bad:
            continue
        if _looks_like_smiles_minimal(c2):
            cleaned.append(c2)

    if not cleaned:
        return ""

    # Score candidates based on presence of SMILES-specific features
    def score(s: str) -> tuple:
        special = sum(ch in "[]=#()@" for ch in s)
        digits = sum(ch.isdigit() for ch in s)
        length = len(s)
        return (special, digits, length)

    cleaned.sort(key=score, reverse=True)
    return cleaned[0]


def generate_smiles_from_caption(
    caption: str,
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt_examples: List[Dict[str, Any]],
    k_shot: int
):
    # 1) Construct the full prompt
    prompt = build_prompt_for_caption(caption, prompt_examples, k_shot=k_shot)

    # 2) Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    # 3) Generate output
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_OUTPUT_LENGTH,
            num_beams=NUM_BEAMS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_len = inputs["input_ids"].shape[1]
    gen_ids = outputs[0][input_len:]
    raw_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    pred_smiles = extract_smiles(raw_text)

    # Return "PARSE_FAIL" if extraction fails
    if pred_smiles == "":
        pred_smiles = "PARSE_FAIL"

    return pred_smiles, raw_text, prompt


# Perform a sanity check on the first caption
pred_smiles_demo, raw_text_demo, demo_prompt = generate_smiles_from_caption(
    captions[2], model, tokenizer, prompt_examples, k_shot=3
)
print("Prompt:\n", demo_prompt[:1600], "...\n")
print("Raw model output:\n", raw_text_demo)
print("Parsed SMILES:\n", pred_smiles_demo)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prompt:
 You are now working as an excellent expert in single-molecule discovery.

Given the caption of a molecule, your job is to predict the SMILES representation of the molecule.
The caption describes the molecule structures using skeleton and anchor groups.
You must infer the correct SMILES representation from the caption.

Output rule:
Output only one SMILES string.
Example 1:
Instruction: Given the caption of a molecule, predict the SMILES representation.
Input:
This molecule has a core structure composed of two benzene rings linked via an azo bond, and contains sulfhydryl groups on both benzene rings, which act as anchor groups.
Output:
C1=CC(=CC=C1N=NC2=CC=C(C=C2)S)S

Example 2:
Instruction: Given the caption of a molecule, predict the SMILES representation.
Input:
This molecule contains a linear carbon chain made up of four carbon atoms with sulfhydryl groups at both terminal ends acting as anchoring moieties.
Output:
C(CCS)CS

Example 3:
Instruction: Given the caption of a mo

In [9]:
# Cell 9: Run few-shot experiments and save results

import os
import pandas as pd
import json

RESULT_DIR = "./Llama3.2-3B-results"
os.makedirs(RESULT_DIR, exist_ok=True)

all_preds_by_k = {}
all_raw_text_by_k = {}

for k in K_SHOTS:
    print(f"\n======= Running few-shot with K = {k} =======")
    random.seed(GLOBAL_RANDOM_SEED)

    records = []

    for i, cap in enumerate(captions):
        print(f"Generating ({k}-shot) {i+1}/{len(captions)} ...", end="\r")

        pred, raw_text, _ = generate_smiles_from_caption(
            cap, model, tokenizer, prompt_examples, k_shot=k
        )

        records.append({
            "index": i,
            "k_shot": k,
            "caption": cap,
            "true_smiles": true_smiles[i],
            "pred_smiles": pred,
            "raw_text": raw_text
        })

    # === Save to CSV ===
    df_k = pd.DataFrame(records)
    csv_path = os.path.join(RESULT_DIR, f"llama3_icl_k{k}.csv")
    df_k.to_csv(csv_path, index=False)

    print(f"\nSaved results to: {csv_path}")

    all_preds_by_k[k] = [r["pred_smiles"] for r in records]
    all_raw_text_by_k[k] = [r["raw_text"] for r in records]

    print(f"\nSample predictions for K={k}:")
    for j in range(min(5, len(records))):
        print(f"[{j}] Caption: {records[j]['caption']}")
        print(f"     True SMILES: {records[j]['true_smiles']}")
        print(f"     Pred SMILES: {records[j]['pred_smiles']}")
        print("-" * 40)

summary_path = os.path.join(RESULT_DIR, "summary.json")
with open(summary_path, "w") as f:
    json.dump(
        {
            "model": "Llama-3.2-3B-Instruct",
            "K_SHOTS": K_SHOTS,
            "num_samples": len(captions),
            "seed": GLOBAL_RANDOM_SEED,
            "result_files": [f"llama3B_icl_k{k}.csv" for k in K_SHOTS]
        },
        f,
        indent=2
    )

print(f"\nExperiment summary saved to: {summary_path}")


======= Running few-shot with K = 0 =======
Generating (0-shot) 60/60 ...
Saved results to: ./Llama3.2-3B-results/llama3_icl_k0.csv

Sample predictions for K=0:
[0] Caption: This molecule has two pyridine cores linked by a diacetylene chain.
     True SMILES: C(C#CC1=CC=NC=C1)#CC1=CC=NC=C1
     Pred SMILES: CC(=O)Nc1ccc(cc1)C(=O)Nc2ccc(cc2)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=O)C(=
----------------------------------------
[1] Caption: This molecule uses a diacetylene chain to link two pyridine rings.
     True SMILES: C(C#CC1=CC=NC=C1)#CC1=CC=NC=C1
     Pred SMILES: CC(=O)C(=O)C1=CC=CC=C1
----------------------------------------
[2] Caption: This molecule connects two pyridine rings via a diacetylene chain.
     True SMILES: C(C#CC1=CC=NC